# Skills 01 · 概念原理与 SKILL 示例

上一章 `07_protocols/` 讲「智能体之间怎么说话」，这一章讲**「怎么把专家知识打包交给智能体」**
—— 也就是 Skills。

| 概念 | 一句话 | 在本节的哪儿 |
|---|---|---|
| Skill | 一个目录 + 一个 `SKILL.md`，就是一个可复用技能包 | 1 节 / 3 节 |
| 三点区别 | 配置成本 / Token 效率 / 维护成本，全都优于「传统 Prompt」 | 1 节的对照表 |
| 渐进式披露 | 选择（只读 name+description）→ 学习（读 SKILL.md 正文）→ 使用（读 references/） | 2 节 + 5.4~5.6 |
| 目录结构 | `SKILL.md` 必需；`scripts/` `references/` `assets/` 三个可选 | 3 节 |
| SKILL.md 八字段 | `name` / `description` / `license` / `compatibility` / `metadata` / `allowed-tools` … | 4 节 |
| 真造一个技能 | 三份内容落盘 + 用标准库 `re` 解析 front-matter + 三步实测 | 5 节 |

> **本 notebook 由 `Agent/_py_source/08_skills/` 下 2 个脚本合并而成**：
> `01_概念与原理_jxsd.py`（463 行：概念 / 原理 / 目录结构 / SKILL.md 四小节）
> 与 `02_SKILL示例_jxsd.py`（540 行：真的造出一个技能包，并实测三步渐进式披露的 Token 账本）。

**官方文档**
- LangChain · Skills（多智能体一章）：<https://docs.langchain.com/oss/python/langchain/multi-agent/skills>
- Deep Agents · Skills：<https://docs.langchain.com/oss/python/deepagents/skills>

## 运行条件

| 项 | 说明 |
|---|---|
| 🟡 运行档位 | **需模型**（本章档位）—— 但**本 notebook 不发起任何模型调用**：它只做概念演示、临时落盘与逐字节比对，纯标准库即可跑通 |
| 依赖 | 无第三方依赖（`re` / `pathlib` 都是标准库） |
| 密钥 | 不需要（不读 `.env` 里的 `api_key` / `base_url`） |
| 前置服务 | 无 |
| 预计耗时 | < 5 秒 |

> **为什么档位标 🟡 却不用模型**：档位是 `08_skills/` 章的约定，真正调模型的是下一本
> `02_三种技能载体.ipynb`（它用 `create_deep_agent(skills=[...])` 去加载技能）。
> 本节按模板要求保留三档标记，但**执行时不碰网络与模型**，所以能直接跑成 PASS。

> **本节需要一份现成的技能资源**：`Agent/08_skills/skills/code-review-skill/`
> （`SKILL.md` + `references/python_rules.md` + `references/javascript_rules.md`）。
> 它**已经作为仓库文件被跟踪**，本节只**读**它 —— 不重新生成、不往章节目录里写文件。

## 本节地图

先把「一个技能从被看到、到被用起来」的全过程画出来，后面每一节都对应这张图上的一个位置。

```mermaid
graph LR
    A["agent 启动<br/>列出技能父目录"] --> B["① 选择<br/>只读 name + description"]
    B --> C{"模型判断<br/>这次要用它吗？"}
    C -->|"否"| D["正文一个字都不进上下文<br/>（省 Token 的来源）"]
    C -->|"是"| E["② 学习<br/>read_file(SKILL.md)"]
    E --> F["③ 使用<br/>按正文里的路由规则<br/>只读命中的那一个 references/*.md"]
    F --> G["产出结论"]
```

上面这张图等价于下面这张表（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 步 | 名字 | 谁来做 | 读进来多少东西 | 本节代码位置 |
|---|---|---|---|---|
| ① | 选择 | 框架（agent 启动时） | 只有 `name` + `description`，几十个字符 | `stage_select()` / 5.4 |
| ② | 学习 | 模型自己决定 | 命中才读 `SKILL.md` 全文（十几行） | `stage_learn()` / 5.5 |
| ③ | 使用 | 模型照正文指示 | 只读命中的那一个 `references/*.md` | `stage_use()` / 5.6 |

**和相邻小节的衔接**

| 小节 | 讲什么 |
|---|---|
| 本节 `01_概念原理与SKILL示例` | 是什么 / 为什么省 Token / 长什么样 / 真造一个并实测三步 |
| 下一本 `02_三种技能载体` | 用 deepagents 加载它、放进 PostgreSQL 做云技能、装进 Claude Code |

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**（这里是 `Agent/08_skills/`），
而本项目所有代码都写 `from config import settings`（`config.py` 在仓库根）。

所以第一格统一做一件事：**向上找到仓库根、切过去、塞进 `sys.path`**，
顺便给出 `NB_DIR`（本 notebook 目录）与 `WORKDIR`（本课临时目录）。

> 本课纯标准库、其实用不到 `config`，但这一格必须保留 —— 一是全仓统一，
> 二是后面的「路径常量」要靠 `NB_DIR` 把源文件里的 `__file__` 顶替掉。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

### 0.1 路径常量：源文件里的 `HERE` 换成 `NB_DIR`

两个源文件都用 `HERE = Path(__file__).resolve().parent` 定位同级资源目录。
**notebook 里没有 `__file__`**（模板第 6 节第 3 条），而 `NB_DIR` 就是
「本 notebook 所在目录」—— 和源文件里的 `HERE` 完全是同一个目录，
所以这里写 `HERE = NB_DIR`，下游所有 `HERE / "skills"` 之类的写法**一行都不用动**。

另外要注意 `WORKDIR` 是**同章共享**的容器（`01_langgraph/` 之类也各有自己的），
所以本课往里写东西时必须再套一层自己的子目录 `skill_build/`，
免得和同章另一本 notebook 并发执行时互相踩文件。

In [ ]:
# ===== 路径常量（源文件里的 HERE 在 notebook 里就是 NB_DIR）=====
HERE = NB_DIR                       # notebook 没有 __file__；NB_DIR 就是本 notebook 所在目录
# Agent/08_skills → Agent → 仓库根
PROJECT_ROOT = HERE.parent.parent
# 技能父目录：deepagents 的 skills 参数要指向「技能的父目录」，就是这一层。
# 这份目录**已经作为仓库文件被跟踪**，本节只读它。
SKILLS_ROOT = HERE / "skills"

SKILL_DIR = SKILLS_ROOT / "code-review-skill"
REFERENCES_DIR = SKILL_DIR / "references"

# notebook 版改写：源文件当年把技能包真的写进章节目录（HERE / "skills"）；
# notebook 每执行一次就会写一次盘，所以改成写进本课专属的临时子目录，
# 再拿它和上面那份已跟踪的技能包做逐字节比对 —— 既不污染章节目录，又保留「落盘」这一步的教学意义。
BUILD_ROOT = WORKDIR / "skill_build"
BUILD_SKILL_DIR = BUILD_ROOT / "code-review-skill"
BUILD_REFERENCES_DIR = BUILD_SKILL_DIR / "references"

print("技能父目录（只读）：", SKILLS_ROOT)
print("本课临时子目录：", BUILD_ROOT)

### 0.2 前置条件自检

本节**不需要密钥、不需要端口、不需要第三方包**，唯一要确认的是
「那份现成的技能包在不在」。缺了就打印中文提示并让后面的「引用现成技能」几步降级跳过，
而不是抛异常 —— 这符合本项目「不抛异常、缺前置就降级」的惯例。

In [ ]:
# ===== 前置条件自检 =====
print("解释器：", sys.executable)

SKILLS_READY = (SKILL_DIR / "SKILL.md").is_file() and REFERENCES_DIR.is_dir()

if SKILLS_READY:
    print("✅ 找到现成的技能包：", SKILL_DIR.name)
    for _p in sorted(SKILL_DIR.rglob("*")):
        if _p.is_file():
            print(f"     {_p.relative_to(SKILLS_ROOT)}   （{len(_p.read_text(encoding='utf-8'))} 字符）")
else:
    print("⚠️ 没找到现成的技能包：", SKILL_DIR)
    print("     后面的「引用现成技能」几步会打印提示并跳过；本 notebook 其余步骤照常可跑。")
    print("     恢复办法：git checkout -- Agent/08_skills/skills")

## 1. 概念：Skills 是 AI 智能体的**插件系统**

课案原文的两句核心论断：

> Skills 本质上是把 AI 智能体封装成一个【可复用的组件】。
> Skills 以 Markdown 文件格式存在，执行功能通过**动态加载**实现。

也就是说，你不是在「写提示词」，而是在**发布一个能力包**：
提示词是散落在各个项目里的一次性文本；技能是有名字、有描述、可以安装/卸载/复用的一个目录。

**Skills 和传统 Prompt 的区别，就三点**（本节最该背下来的东西）：

| 维度 | 传统 Prompt | Skills |
|---|---|---|
| 配置成本 | 每次使用都要重新配置、复制粘贴一遍提示词 | 仅需一次配置，后续可永久复用；放在技能目录里谁都能装 |
| Token 效率 | 每次调用**全量加载**内容，提示词越长占用越高 | 按需懒加载：平时只看到名字 + 描述，匹配到才加载正文 |
| 维护成本 | 跨场景、跨项目反复复制粘贴，维护繁琐易出错 | 只改 `SKILL.md` 一个文件，即可全局/项目级统一生效 |

下面这一格先把「中文等宽对齐」的三个小工具准备好 —— 源文件里的表格是打印到控制台的，
中文是全角字符占 2 列，不做宽度换算，用 `str.ljust()` 排出来的表一定是歪的。

In [ ]:
# ================================================================
# 小工具：让中文表格在控制台里对齐
# ================================================================
def display_width(text: str) -> int:
    """计算字符串在等宽终端里占多少「列」。

    中文、中文标点是全角字符，占 2 列；ASCII 占 1 列。
    不处理这个，用 str.ljust() 排出来的中文表格会歪掉。
    """
    width = 0
    for ch in text:
        # 覆盖常用中日韩统一表意文字与常见全角标点区段，够本项目表格使用。
        # 只判「是不是全角」而不做完整 Unicode 宽度表，是因为表格里的字符
        # 全部来自本文件的常量，范围可控；真要做通用终端宽度，得用 wcwidth 那类表。
        if "\u1100" <= ch <= "\u115f" or "\u2e80" <= ch <= "\ua4cf" or "\uac00" <= ch <= "\ud7a3":
            width += 2
        elif "\uf900" <= ch <= "\ufaff" or "\ufe30" <= ch <= "\ufe6f" or "\uff00" <= ch <= "\uff60":
            width += 2
        else:
            width += 1
    return width


def pad(text: str, width: int) -> str:
    """按「显示宽度」右侧补空格，用于打印中文表格。"""
    return text + " " * max(0, width - display_width(text))


def print_table(headers: list[str], rows: list[list[str]], widths: list[int]) -> None:
    """打印一张宽度可控的表格（分隔线用 ASCII，避免不同终端渲染差异）。

    单元格里允许出现 `\\n`，会被拆成多行显示（同一行的其它列补空）。
    """
    line = "+" + "+".join("-" * (w + 2) for w in widths) + "+"
    print(line)
    print("| " + " | ".join(pad(h, w) for h, w in zip(headers, widths)) + " |")
    print(line)
    for row in rows:
        # 每个单元格按 \n 拆成多行，整行高度取最大值
        cells = [c.split("\n") for c in row]
        for i in range(max(len(c) for c in cells)):
            parts = [pad(c[i] if i < len(c) else "", w) for c, w in zip(cells, widths)]
            print("| " + " | ".join(parts) + " |")
        print(line)

### 1.1 第 1 节：概念 + 三点区别对照表

这一格就是源文件 `01_概念与原理_jxsd.py` 的 `section_concept()`：
先念一遍课案原文，再把「三点区别」逐条落成一张等宽对齐的表。

In [ ]:
# ================================================================
# 1. 概念：Skills 是什么，它和「传统 Prompt」差在哪
# ================================================================
def section_concept() -> None:
    print("=" * 78)
    print("1. 概念：Skills 是 AI 智能体的插件系统，封装特定应用场景的专家知识")
    print("=" * 78)
    print(
        """
课案原文要点：
    Skills 本质上是把 AI 智能体封装成一个【可复用的组件】。
    Skills 以 Markdown 文件格式存在，执行功能通过动态加载实现。

也就是说，你不是在「写提示词」，而是在「发布一个能力包」：
    提示词是散落在各个项目里的一次性文本；
    技能是有名字、有描述、有版本、可以安装/卸载/复用的一个目录。
"""
    )

    # 课案给出的三点区别，逐条落成表；这三条是本节最该背下来的东西。
    print("【Skills 和传统 Prompt 的区别】三点：")
    rows = [
        [
            "配置成本",
            "每次使用都要重新配置，\n复制粘贴一遍提示词",
            "仅需一次配置，后续可永久复用；\n放在技能目录里谁都能装",
        ],
        [
            "Token 效率",
            "每次调用【全量加载】内容，\n提示词越长占用越高",
            "按需懒加载：平时只看到名字+描述，\n匹配到才加载正文，大幅降低消耗",
        ],
        [
            "维护成本",
            "需跨场景、跨项目反复复制粘贴，\n维护繁琐易出错",
            "只改 SKILL.md 一个文件，\n即可实现全局/项目级统一生效",
        ],
    ]
    # 单元格里有 \n，这里按行拆开打印，保证等宽对齐
    headers = ["维度", "传统 Prompt", "Skills"]
    widths = [10, 30, 32]
    line = "+" + "+".join("-" * (w + 2) for w in widths) + "+"
    print(line)
    print("| " + " | ".join(pad(h, w) for h, w in zip(headers, widths)) + " |")
    print(line)
    for row in rows:
        cells = [c.split("\n") for c in row]
        height = max(len(c) for c in cells)
        for i in range(height):
            parts = [pad(c[i] if i < len(c) else "", w) for c, w in zip(cells, widths)]
            print("| " + " | ".join(parts) + " |")
        print(line)


section_concept()

### 预期输出

```text
==============================================================================
1. 概念：Skills 是 AI 智能体的插件系统，封装特定应用场景的专家知识
==============================================================================

课案原文要点：
    Skills 本质上是把 AI 智能体封装成一个【可复用的组件】。
    Skills 以 Markdown 文件格式存在，执行功能通过动态加载实现。

也就是说，你不是在「写提示词」，而是在「发布一个能力包」：
    提示词是散落在各个项目里的一次性文本；
    技能是有名字、有描述、有版本、可以安装/卸载/复用的一个目录。

【Skills 和传统 Prompt 的区别】三点：
+------------+--------------------------------+----------------------------------+
| 维度       | 传统 Prompt                    | Skills                           |
+------------+--------------------------------+----------------------------------+
| 配置成本   | 每次使用都要重新配置，         | 仅需一次配置，后续可永久复用；   |
|            | 复制粘贴一遍提示词             | 放在技能目录里谁都能装           |
+------------+--------------------------------+----------------------------------+
| Token 效率 | 每次调用【全量加载】内容，     | 按需懒加载：平时只看到名字+描述， |
|            | 提示词越长占用越高             | 匹配到才加载正文，大幅降低消耗   |
+------------+--------------------------------+----------------------------------+
| 维护成本   | 需跨场景、跨项目反复复制粘贴， | 只改 SKILL.md 一个文件，         |
|            | 维护繁琐易出错                 | 即可实现全局/项目级统一生效      |
+------------+--------------------------------+----------------------------------+
```

这张表是**打印出来的**，所以中文列宽是算过的（`display_width`）——
左边 `传统 Prompt` 那一列为什么歪不了，就是上面那个小工具在起作用。

## 2. 原理：一个文件夹 + 一个 `SKILL.md`，靠「逐步加载」省 Token

课案原文要点：

> Skills 的核心就是：一个文件夹 + 一个 `SKILL.md` 文件。

**逐步加载（渐进式披露）三步**：

1. **选择**：AI 只读取每个技能的名称和描述（只有几十个字符）；
2. **学习**：匹配到某个技能时，AI 才把该 `SKILL.md` 里的内容加载进来；
3. **使用**：AI 按照指示执行，可参考其他文件、写代码等。

对应到 deepagents 的实现（`SkillsMiddleware`，本机 0.7.13 实测）：

| 步 | 真实框架里发生的事 |
|---|---|
| 选择 | agent 启动时中间件 `ls` 技能父目录 → 对每个子目录下载 `SKILL.md` → **只解析 YAML front-matter**，把 `name`/`description` 拼进系统提示词 |
| 学习 | 模型决定用这个技能时，它自己发起 `read_file`，把 `SKILL.md` 全文（含「去读 references/xxx.md」这类指令）拉进上下文 |
| 使用 | 模型按 `SKILL.md` 的指示干活：读参考文件、跑 `scripts/` 里的脚本 |

课案举的例子是一个「代码审查」技能：`SKILL.md` 只有 10 行（充当**路由器**），
三份语言规范全放在 `references/` 里，各 200~300 行。
用户说「帮我审查这段 Python 代码」时，**只需要加载 Python 规范**，
JavaScript 与 C++ 那 500 行永远不进上下文 —— 这就是「既快又省」的来源。

下面这一格把嵌套 dict 渲染成 `tree` 风格的文本行，用来画这些目录树。

In [ ]:
def render_tree(node: dict, prefix: str = "") -> list[str]:
    """把嵌套 dict 渲染成 `tree` 风格的文本行。

    约定（三种取值）：
        dict            → 目录，递归展开
        str             → 文件，字符串就是行尾注释
        ("dir", "注释") → 带注释的目录（不展开）
    """
    lines: list[str] = []
    items = list(node.items())
    for index, (name, payload) in enumerate(items):
        is_last = index == len(items) - 1
        branch = "└── " if is_last else "├── "
        if isinstance(payload, dict):
            lines.append(f"{prefix}{branch}{name}/")
            # 子节点前缀：最后一个用空格续行，其余用竖线续行
            lines.extend(render_tree(payload, prefix + ("    " if is_last else "│   ")))
        elif isinstance(payload, tuple):
            # 带注释的目录。注释可能有多行，续行要缩进到「注释起始列」：
            # 前缀(4) + 名字宽度 + "/"(1) + "    # "(6)
            _kind, note = payload
            note_lines = note.split("\n")
            lines.append(f"{prefix}{branch}{name}/    # {note_lines[0]}")
            # 注释换行后要对齐到 "#" 后面的那一列，否则树形结构看起来是断的；
            # 这里用 display_width(name) 而不是 len(name)，就是因为中文目录名占 2 列。
            cont = prefix + ("    " if is_last else "│   ")
            cont += " " * (display_width(name) + 1 + 6)
            lines.extend(f"{cont}{extra}" for extra in note_lines[1:])
        else:
            suffix = f"    # {payload}" if payload else ""
            lines.append(f"{prefix}{branch}{name}{suffix}")
    return lines

### 2.1 第 2 节：逐步加载机制

这一格是源文件里的 `section_principle()`：画出课案那个「代码审查」技能目录，
再算一笔 Token 账 —— 全量塞进提示词 vs 逐步加载，差多少行。

In [ ]:
# ================================================================
# 2. 原理：逐步加载（渐进式披露）三步走
# ================================================================
def section_principle() -> None:
    print()
    print("=" * 78)
    print("2. 原理：一个文件夹 + 一个 SKILL.md，靠「逐步加载」省 Token")
    print("=" * 78)
    print(
        """
课案原文要点：
    Skills 的核心就是：一个文件夹 + 一个 SKILL.md 文件。

【逐步加载机制】三步：
    1. 选择：AI 只读取每个技能的名称和描述（只有几十字符）
    2. 学习：当匹配到某个技能时，AI 才会把该 SKILL.md 里的内容加载进来
    3. 使用：AI 按照指示执行，可参考其他文件、写代码等

对应到 deepagents 的实现（SkillsMiddleware，本机 0.7.13 实测）：
    第 1 步发生在 agent 启动时 —— 中间件 ls 技能父目录，对每个子目录
        下载 SKILL.md、只解析 YAML front-matter，把 name/description
        拼进系统提示词（所以「名字+描述」常驻上下文，正文不常驻）；
    第 2 步发生在模型决定用这个技能时 —— 它自己发起 read_file，
        把 SKILL.md 全文（含"去读 references/xxx.md"这类指令）拉进上下文；
    第 3 步是模型按 SKILL.md 的指示干活：读参考文件、跑 scripts/ 里的脚本。
"""
    )

    print("【为什么要逐步加载？】课案给的例子 —— 一个「代码审查」Skill：")
    tree = {
        "code-review-skill": {
            "SKILL.md": "只有 10 行，充当『路由器』",
            "references": {
                "python_rules.md": "200 行",
                "javascript_rules.md": "200 行",
                "cpp_rules.md": "300 行",
            },
        }
    }
    for line in render_tree(tree):
        print("  " + line)

    print(
        """
课案原文结论：
    如果用户说「帮我审查这段 Python 代码」，AI 只需要加载 Python 规范部分，
    不需要加载 JavaScript 规范和 C++ 规范。
    这样【既能让 AI 快速响应，同时还能节省 Token】。

算一笔账（课案例子的极端情况）：
    全量塞进提示词  = 10 + 200 + 200 + 300 = 710 行，每一轮对话都要带；
    逐步加载        = 常驻 10 行（其实是 name+description 几十字符）
                    + 命中后读 10 行 SKILL.md
                    + 真正需要的 200 行 python_rules.md
                    ≈ 220 行，且 javascript/cpp 那 500 行永远不进上下文。
    技能越多、参考文件越长，这个差距越大 —— 这是 Skills 相对「把资料全写进
    system prompt」最大的工程价值。
"""
    )


section_principle()

### 预期输出

```text
==============================================================================
2. 原理：一个文件夹 + 一个 SKILL.md，靠「逐步加载」省 Token
==============================================================================

课案原文要点：
    Skills 的核心就是：一个文件夹 + 一个 SKILL.md 文件。

【逐步加载机制】三步：
    1. 选择：AI 只读取每个技能的名称和描述（只有几十字符）
    2. 学习：当匹配到某个技能时，AI 才会把该 SKILL.md 里的内容加载进来
    3. 使用：AI 按照指示执行，可参考其他文件、写代码等

对应到 deepagents 的实现（SkillsMiddleware，本机 0.7.13 实测）：
    第 1 步发生在 agent 启动时 —— 中间件 ls 技能父目录，对每个子目录
        下载 SKILL.md、只解析 YAML front-matter，把 name/description
        拼进系统提示词（所以「名字+描述」常驻上下文，正文不常驻）；
    第 2 步发生在模型决定用这个技能时 —— 它自己发起 read_file，
        把 SKILL.md 全文（含"去读 references/xxx.md"这类指令）拉进上下文；
    第 3 步是模型按 SKILL.md 的指示干活：读参考文件、跑 scripts/ 里的脚本。

【为什么要逐步加载？】课案给的例子 —— 一个「代码审查」Skill：
  └── code-review-skill/
      ├── SKILL.md    # 只有 10 行，充当『路由器』
      └── references/
          ├── python_rules.md    # 200 行
          ├── javascript_rules.md    # 200 行
          └── cpp_rules.md    # 300 行

课案原文结论：
    如果用户说「帮我审查这段 Python 代码」，AI 只需要加载 Python 规范部分，
    不需要加载 JavaScript 规范和 C++ 规范。
    这样【既能让 AI 快速响应，同时还能节省 Token】。

算一笔账（课案例子的极端情况）：
    全量塞进提示词  = 10 + 200 + 200 + 300 = 710 行，每一轮对话都要带；
    逐步加载        = 常驻 10 行（其实是 name+description 几十字符）
                    + 命中后读 10 行 SKILL.md
                    + 真正需要的 200 行 python_rules.md
                    ≈ 220 行，且 javascript/cpp 那 500 行永远不进上下文。
    技能越多、参考文件越长，这个差距越大 —— 这是 Skills 相对「把资料全写进
    system prompt」最大的工程价值。
```

课案那个 `cpp_rules.md` 在本仓库的技能包里**故意没有**：仓库里只保留
`python_rules.md` 与 `javascript_rules.md` 两份 —— 少一份参考文件，
「只读命中的那一份」这件事在第 5 节才好数得清。

## 3. 目录结构：最小结构 vs 完整结构

课案原文要点：

> 一个 Skill 包含：
> - **元数据**：必须包含名称、描述；
> - **指令**：技能的详细指示。

| 结构 | 内容 |
|---|---|
| **最小结构** | 一个目录 + 一个 `SKILL.md`（唯一必需文件） |
| **完整结构（可选）** | 再加 `scripts/`、`references/`、`assets/` 三个目录 |

三个可选目录的分工 —— 记住这个划分，写技能时就不会乱放东西：

| 目录 | 放什么 | 谁来读 |
|---|---|---|
| `scripts/` | 可执行的代码。`SKILL.md` 里写「运行 `scripts/convert.py`」 | 让 agent 用 shell 去**跑**，而不是把代码读进上下文 |
| `references/` | 只在需要时才读的文档（规范、语法手册、检查清单） | 模型按需 `read_file` —— **渐进式披露的主战场** |
| `assets/` | 模板文件、示例图片、默认配置（比如默认输出 PNG 还是 SVG） | 不是给人读的，是给脚本/模型**用**的 |

下面这一格 `print_real_tree()` 会把**磁盘上真实存在的**目录树打印出来
（源文件里是给 `02_SKILL示例_jxsd.py` 生成的目录用的，这里用来观察第 5 节落盘的结果）。

In [ ]:
def print_real_tree(root: Path, max_depth: int = 3) -> None:
    """打印磁盘上真实存在的目录树（用于观察 02 小节生成的结果）。"""
    if not root.exists():
        print(f"  （{root} 还不存在 —— 本 notebook 第 5 节会在临时目录里重放一份）")
        return

    def walk(path: Path, prefix: str = "", depth: int = 0) -> None:
        if depth > max_depth:
            return
        # 排序键 (p.is_file(), p.name)：先目录后文件，同类按名字排。
        # 不排的话 iterdir() 的顺序依赖文件系统，每次打印出来的树都可能不一样。
        children = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name))
        for index, child in enumerate(children):
            is_last = index == len(children) - 1
            branch = "└── " if is_last else "├── "
            mark = "/" if child.is_dir() else ""
            size = f"    # {child.stat().st_size} 字节" if child.is_file() else ""
            print(f"  {prefix}{branch}{child.name}{mark}{size}")
            if child.is_dir():
                walk(child, prefix + ("    " if is_last else "│   "), depth + 1)

    print(f"  {root.name}/")
    walk(root)

### 3.1 第 3 节：两种结构 + 本仓库里真实的技能目录

这一格是源文件里的 `section_layout()`：先画「最小结构」与「完整结构」，
最后一行 `print_real_tree(SKILLS_ROOT)` 打印的就是**仓库里那份已经跟踪的技能包**
（`Agent/08_skills/skills/`）—— 也就是说，课案讲的结构在这里是**真的存在**的。

In [ ]:
# ================================================================
# 3. 目录结构：最小结构 vs 完整结构
# ================================================================
def section_layout() -> None:
    print()
    print("=" * 78)
    print("3. 目录结构：一个 Skill 包含「元数据」+「指令」")
    print("=" * 78)
    print(
        """
课案原文要点：
    一个 Skill 包含：
        - 元数据：必须包含名称、描述
        - 指令：技能的详细指示
"""
    )

    print("【最小结构】—— 只有一个必需文件：")
    for line in render_tree({"image-to-mermaid": {"SKILL.md": "唯一必需文件"}}):
        print("  " + line)

    print()
    print("【完整结构（可选）】—— 目录名就是技能名，四个位置各司其职：")
    full = {
        "image-to-mermaid": {
            "SKILL.md": "必需：指令 + 元数据",
            "scripts": ("dir", "可选：执行脚本"),
            "references": ("dir", "可选：文档资料，渐进式披露，\n如下例的 mermaid 语法规范"),
            "assets": ("dir", "可选：资源（模板文件、示例图片、配置项，\n比如默认输出 PNG 还是 SVG）"),
        }
    }
    for line in render_tree(full):
        print("  " + line)

    print(
        """
三个可选目录的分工（记住这个划分，写技能时就不会乱放东西）：
    scripts/     —— 可执行的代码。SKILL.md 里写「运行 scripts/convert.py」，
                    让 agent 用 shell 去跑，而不是把代码读进上下文。
    references/  —— 只在需要时才读的文档（规范、语法手册、检查清单）。
                    这是「渐进式披露」的主战场。
    assets/      —— 不是给人读的，是给脚本/模型用的资源：模板、示例图、
                    默认配置（比如默认输出 PNG 还是 SVG）。
"""
    )

    print("【本仓库里真实的技能目录】—— 已跟踪的 Agent/08_skills/skills/，本 notebook 只读它：")
    print_real_tree(SKILLS_ROOT)


section_layout()

### 预期输出

```text
==============================================================================
3. 目录结构：一个 Skill 包含「元数据」+「指令」
==============================================================================

课案原文要点：
    一个 Skill 包含：
        - 元数据：必须包含名称、描述
        - 指令：技能的详细指示

【最小结构】—— 只有一个必需文件：
  └── image-to-mermaid/
      └── SKILL.md    # 唯一必需文件

【完整结构（可选）】—— 目录名就是技能名，四个位置各司其职：
  └── image-to-mermaid/
      ├── SKILL.md    # 必需：指令 + 元数据
      ├── scripts/    # 可选：执行脚本
      ├── references/    # 可选：文档资料，渐进式披露，
      │                    如下例的 mermaid 语法规范
      └── assets/    # 可选：资源（模板文件、示例图片、配置项，
                       比如默认输出 PNG 还是 SVG）

三个可选目录的分工（记住这个划分，写技能时就不会乱放东西）：
    scripts/     —— 可执行的代码。SKILL.md 里写「运行 scripts/convert.py」，
                    让 agent 用 shell 去跑，而不是把代码读进上下文。
    references/  —— 只在需要时才读的文档（规范、语法手册、检查清单）。
                    这是「渐进式披露」的主战场。
    assets/      —— 不是给人读的，是给脚本/模型用的资源：模板、示例图、
                    默认配置（比如默认输出 PNG 还是 SVG）。

【本仓库里真实的技能目录】—— 已跟踪的 Agent/08_skills/skills/，本 notebook 只读它：
  skills/
  └── code-review-skill/
      ├── references/
      │   ├── javascript_rules.md    # 2527 字节
      │   └── python_rules.md    # 3213 字节
      └── SKILL.md    # 1395 字节
```

**两处值得对照着看的细节**：

1. 最后那棵是真·磁盘上的树（`print_real_tree`），**不是画出来的常量** ——
   `references/` 里正好两份规范，与课案说的「按语言分文件」一致；
2. 目录树里的字节数 ≠ 字符数：`SKILL.md` 是 1395 字节 / 783 字符
   （中文一个字 3 字节，`.read_text()` 数出来的才是字符）。
   第 5 节的 Token 账本一律按**字符**算，别和字节混。

## 4. `SKILL.md`：基本模板 + 元数据字段表

`SKILL.md` 顶部是一段 **YAML front-matter**（用两行 `---` 夹住），里面写元数据；
下面正文是 Markdown 格式的指令，智能体会在**选择该技能时**读取它。

```markdown
---
name: skill-name
description: 说明这个 Skill 的功能以及使用场景
---

# 在这里开始写你的内容，Markdown 格式 —— 智能体会在选择该技能时读取
```

**两个容易踩的坑**：

1. `---` 必须是文件的**第 1 行**，前后不能有空行；正文中间**不能出现单独一行的 `---`**
   —— 中间那行会被当成 front-matter 的结束符，正文就被吃掉了；
2. `name` 必须和技能**所在目录名**完全一致。deepagents 会校验这一点，
   不一致时在日志里警告 `name 'x' must match directory name 'y'`。

下面是课案《元数据字段说明》表格原样搬过来的八个字段（第 8 个 `path` 是
本机 deepagents 0.7.13 的 `SkillMetadata` 里多出来的运行时字段，一并列出方便和源码对照）：

In [ ]:
# ================================================================
# 4. SKILL.md：基本模板 + 元数据字段表
# ================================================================
SKILL_MD_TEMPLATE = """---
name: skill-name
description: 说明这个 Skill 的功能以及使用场景
---

# 在这里开始写你的内容，Markdown 格式 —— 智能体会在选择该技能时读取
"""

# 课案《元数据字段说明》表格，逐字段抄下来。
# 本机 deepagents 0.7.13 的 SkillMetadata 还多一个 path（由中间件运行时注入，
# 不需要你手写），一并列在第 8 行，方便和源码对照。
METADATA_FIELDS = [
    ["name", "是", "Skill 名称，最长 64 字符，只允许使用小写字母、数字和 -，\n也不能以 - 开头或结尾"],
    ["description", "是", "简短说明使用场景，最长 1024 字符，不能为空"],
    ["trigger_keywords", "否", "强制推荐关键词，自动触发时使用"],
    ["license", "否", "开源许可证，或指向 Skill 附带的许可证文件"],
    ["compatibility", "否", "描述兼容性，说明与哪些产品系统、平台权限等有关，\n最长 500 字符"],
    ["metadata", "否", "自定义键值对，用于扩展元数据，如作者、版本号等"],
    ["allowed-tools", "否", "允许使用的工具列表，空格分隔的已有工具或新工具功能"],
    ["path", "—", "运行时注入，由中间件写入 SKILL.md 的路径。\n你不写它，但 read_file 时用的就是它"],
]

### 4.1 第 4 节：模板 + 字段表 + 逐字段补充说明

In [ ]:
def section_skill_md() -> None:
    print()
    print("=" * 78)
    print("4. SKILL.md：基本模板 + 元数据字段表")
    print("=" * 78)
    print(
        """
课案原文要点：
    SKILL.md 顶部是一段 YAML front-matter（用两行 `---` 夹住），
    里面写元数据；下面正文是 Markdown 格式的指令，
    智能体会在【选择该技能时】读取它。
"""
    )

    print("【基本模板】（课案原文，可直接复制改名使用）：")
    print("-" * 78)
    print(SKILL_MD_TEMPLATE, end="")
    print("-" * 78)
    print(
        """
两个容易踩的坑：
    1. `---` 必须是文件的第 1 行，且前后不能有空行；中间不能出现单独一行的 `---`
       （中间那行会被当成 front-matter 的结束符，正文就被吃掉了）。
    2. name 必须和技能【所在目录名】完全一致。本机 deepagents 会校验这一点，
       不一致时会在日志里警告 `name 'x' must match directory name 'y'`。
"""
    )

    print()
    print("【元数据字段表】八个字段，逐字段解释：")
    print_table(
        ["字段", "必需", "说明"],
        METADATA_FIELDS,
        [18, 6, 62],
    )
    print(
        """
逐字段补充说明（课案表格之外，写的时候容易含糊的地方）：

    name            技能的唯一标识，也是模型在系统提示词里看到的键。
                    只允许小写字母 / 数字 / 单个连字符，不能有连续 `--`，
                    且必须等于 SKILL.md 所在目录的名字。
    description     最关键的字段 —— 模型【只靠它】决定要不要用这个技能。
                    写法 = 「做什么」+「什么时候用」+「关键触发词」，
                    例如：「Python 代码审查，检查规范性和性能问题；
                    当用户要求 review / 检查 / 优化 Python 代码时使用」。
    trigger_keywords强制推荐关键词，用于自动触发。
                    注意：本机 deepagents 0.7.13 的 SkillsMiddleware **不读**
                    这个字段（它只解析 name/description/license/compatibility/
                    metadata/allowed-tools）。写进去不报错，但当前版本不会生效——
                    所以真正想提高命中率，还得把关键词写进 description。
    license         开源许可证名（MIT / Apache-2.0）或指向附带的 LICENSE 文件。
    compatibility   环境要求。例如「需要 uv、Python 3.11+、可访问外网」，
                    或「仅在 Claude Code 中可用」。
    metadata        自定义键值对。约定俗成放 author / version / updated_at。
    allowed-tools   建议该技能使用哪些工具，空格分隔（也接受 YAML 列表）。
                    本机 deepagents 会把它渲染成
                    `-> Allowed tools: read_file, write_file` 一行提示。
                    官方标注为 experimental（实验特性）。
    path            运行时字段：中间件把 SKILL.md 的真实路径塞进来，
                    模型随后用这个路径调 read_file 拉全文。你不需要手写。

完整 front-matter 长这样（把下面这段抄进 SKILL.md 就是一份合规的元数据）：

    ---
    name: code-review-skill
    description: Python/JavaScript 代码审查，检查规范性与性能问题
    trigger_keywords: 代码审查 review 规范 性能
    license: MIT
    compatibility: 需要 Python 3.10+ 与可读写的项目目录
    metadata:
      author: jxsd
      version: "1.0.0"
    allowed-tools: read_file write_file execute
    ---
"""
    )


section_skill_md()

### 预期输出

```text
==============================================================================
4. SKILL.md：基本模板 + 元数据字段表
==============================================================================

课案原文要点：
    SKILL.md 顶部是一段 YAML front-matter（用两行 `---` 夹住），
    里面写元数据；下面正文是 Markdown 格式的指令，
    智能体会在【选择该技能时】读取它。

【基本模板】（课案原文，可直接复制改名使用）：
------------------------------------------------------------------------------
---
name: skill-name
description: 说明这个 Skill 的功能以及使用场景
---

# 在这里开始写你的内容，Markdown 格式 —— 智能体会在选择该技能时读取
------------------------------------------------------------------------------

两个容易踩的坑：
    1. `---` 必须是文件的第 1 行，且前后不能有空行；中间不能出现单独一行的 `---`
       （中间那行会被当成 front-matter 的结束符，正文就被吃掉了）。
    2. name 必须和技能【所在目录名】完全一致。本机 deepagents 会校验这一点，
       不一致时会在日志里警告 `name 'x' must match directory name 'y'`。


【元数据字段表】八个字段，逐字段解释：
+--------------------+--------+----------------------------------------------------------------+
| 字段               | 必需   | 说明                                                           |
+--------------------+--------+----------------------------------------------------------------+
| name               | 是     | Skill 名称，最长 64 字符，只允许使用小写字母、数字和 -，       |
|                    |        | 也不能以 - 开头或结尾                                          |
+--------------------+--------+----------------------------------------------------------------+
| description        | 是     | 简短说明使用场景，最长 1024 字符，不能为空                     |
+--------------------+--------+----------------------------------------------------------------+

……（中间 6 张同格式的字段行与「逐字段补充说明」整段略去 ——
    与第 4 节上面的字段表、补充说明逐字一致；本格完整输出 4317 字符）

完整 front-matter 长这样（把下面这段抄进 SKILL.md 就是一份合规的元数据）：

    ---
    name: code-review-skill
    description: Python/JavaScript 代码审查，检查规范性与性能问题
    trigger_keywords: 代码审查 review 规范 性能
    license: MIT
    compatibility: 需要 Python 3.10+ 与可读写的项目目录
    metadata:
      author: jxsd
      version: "1.0.0"
    allowed-tools: read_file write_file execute
    ---
```

上面那段 front-matter 就是第 5 节要落盘的真实内容 —— **八字段里用到了六个**，
没用到的是 `path`（运行时注入，不用你写）。

> ⚠️ 上面是**节选**：中间 6 张同格式的字段行与「逐字段补充说明」整段，
> 与前面第 4 节的字段表逐字一致，这里用「……」略去了（本格完整输出 4317 字符）。
> 所以**别逐字比对整段**，该核的是「表格格式 + front-matter 八字段」这个结构。

## 5. 实战：在磁盘上生成一个真能用的技能，并实测「渐进式披露」

前面四节都是「讲」，第 5 节开始「造」。源文件 `02_SKILL示例_jxsd.py` 做了三件事：

1. **真的造出**一个技能目录（不是讲概念）；
2. 把「选择 → 学习 → 使用」三步各跑一遍，每一步都打印**字符数 / 行数**，
   让「省 Token」从一句话变成一个看得见的数字；
3. 用标准库 `re` 手写一个十几行的 front-matter 解析器，并讲清它**不支持**什么。

技能包的形状：

```text
code-review-skill/
├── SKILL.md                    ← 10 来行的「路由器」
└── references/
    ├── python_rules.md         ← 只在审查 Python 时才读
    └── javascript_rules.md     ← 只在审查 JavaScript 时才读
```

> **notebook 版的一处改写（重要）**：源文件当年是**真的往 `Agent/08_skills/skills/` 写盘**的；
> notebook 每次执行都会写一次，会反复往章节目录里落文件。所以本 notebook 把「落盘」改到
> **本课专属临时目录** `Agent/08_skills/tmp_nb_work/skill_build/`，
> 再把写出来的三个文件与仓库里**已跟踪的**那份**逐字节比对**（应当完全一致）。
> 这样既保住了「写盘」这一步的教学意义，又不动仓库里那份现成技能。

**为什么不用 PyYAML**：规范第 1 节铁律要求不装新依赖；而 front-matter 是个
极小的 YAML 子集（只有顶层 `key: value` 和一层缩进的 map），用标准库 `re` 手写十几行就够。

### 5.1 技能包要落盘的三份内容

下面三个字符串常量就是技能包的**全部内容**（`SKILL.md` 一份 + 两份语言规范）。

注意三点：

1. `SKILL.md` 里的 `---` 必须是第 1 行；
2. `name: code-review-skill` 必须和所在目录名完全一致；
3. **正文要短** —— 它就是课案说的「只有 10 行，充当路由器」：
   正文只负责告诉模型「什么语言去读哪个文件」，规范细节全放 `references/`。

还注意 `references/` 里那两份规范**必须分文件放**：合成一份的话，
「只读命中的那一份」就无从谈起，渐进式披露也就没了。

In [ ]:
# ================================================================
# 0. 准备要写入磁盘的技能内容
# ================================================================
# SKILL.md：注意三点 ——
#   1. `---` 必须是第 1 行；
#   2. name 必须和所在目录名（code-review-skill）完全一致；
#   3. 正文要短，它就是课案说的「只有 10 行，充当路由器」——
#      正文里只负责告诉模型「什么语言去读哪个文件」，规范细节全放 references/。
SKILL_MD = """---
name: code-review-skill
description: 审查 Python / JavaScript 代码，检查规范性、性能与安全问题。当用户要求「代码审查 / review / 检查代码 / 优化这段代码」时使用。
trigger_keywords: 代码审查 review 检查代码 规范 性能 安全
license: MIT
compatibility: 需要能读取项目源码目录；Python 3.10+
metadata:
  author: jxsd
  version: "1.0.0"
allowed-tools: read_file write_file
---

# 代码审查技能

你是资深代码审查专家。收到审查请求后，严格按下面的流程执行。

## 第 1 步：判断语言

- 代码是 Python（`.py`，或含 `def` / `import` / 缩进块）→ 读 `references/python_rules.md`
- 代码是 JavaScript / TypeScript（`.js` `.ts`，或含 `const` / `=>` / `function`）→ 读 `references/javascript_rules.md`
- 混合项目 → 两种规范都读，分别给出结论

**只读命中的那一份规范**，不要为了「保险」把两份都读进来。

## 第 2 步：逐条比对

按规范的编号逐条检查，每条给出三样东西：

1. 结论：通过 / 不通过 / 存疑
2. 证据：`文件名:行号` + 原始代码片段
3. 修法：可直接替换的改法，不要写「建议优化」这种空话

## 第 3 步：汇总

输出一张表（严重 / 一般 / 建议三档），最后给一句总体结论。

不要重写整个文件，只给需要改的片段。
"""

# references/python_rules.md —— 课案里说这类规范「200 行」，这里写一份精简可用的版本。
PYTHON_RULES_MD = """# Python 代码审查规范（33 条）

> 本文件只在审查 Python 代码时才需要读入上下文 —— 这就是「渐进式披露」。

## 一、命名与风格（PEP 8）

1. 模块名 `snake_case`，且尽量短；不要用 `utils.py` 这种什么都往里塞的名字。
2. 类名 `PascalCase`；函数与变量 `snake_case`；常量 `UPPER_SNAKE_CASE`。
3. 私有成员用单下划线前缀 `_helper`；不要用双下划线做「伪私有」。
4. 单行不超过 100 字符（本项目约定）；续行用括号隐式拼接，不要用反斜杠。
5. import 分三组：标准库 / 第三方 / 本项目，组间空一行。
6. 不要 `from x import *`；它会让静态检查失效。
7. 函数之间空两行，类的方法之间空一行。
8. 注释写「为什么」，不复述「做了什么」。

## 二、类型注解

9. 公开函数必须有参数与返回值注解。
10. 用 `list[str]` / `dict[str, int]`（PEP 585 内置泛型），不要 `List` / `Dict`。
11. 允许 `None` 的用 `X | None`，不要用 `Optional[X]`。
12. 用 `Sequence` / `Iterable` 做参数类型，用 `list` 做返回值类型（里氏替换）。
13. 复杂结构用 `TypedDict` 或 `dataclass`，不要一路嵌套 `dict[str, Any]`。

## 三、异常处理

14. 禁止裸 `except:`；至少 `except Exception:`。
15. `except` 后必须做点什么：记录日志、重新抛出、或返回明确的兜底值。
16. 不要用异常做流程控制（比如用 `try/except KeyError` 代替 `if key in d`）。
17. 抛业务异常要带上下文：`raise ValueError(f"uid={uid} 不存在") from e`。
18. 资源获取一律用 `with`（文件、连接、锁）；不要手写 `try/finally` 关文件。

## 四、性能

19. 循环里不要做重复计算：把 `len(x)`、属性查找提到循环外。
20. 拼接大量字符串用 `"".join(parts)`，不要 `s += piece`。
21. 成员判断用 `set` / `dict`，不要用 `list` 做 `in`（O(n) → O(1)）。
22. 生成器优先：能用 `(x for x in ...)` 就不要先建一个 list。
23. 不要过早优化 —— 先测量（`timeit` / `cProfile`），再动手。

## 五、安全

24. 禁止 `eval()` / `exec()` 处理外部输入。
25. 拼接 SQL 一律用参数化查询，不要 f-string 拼表名、字段名。
26. `pickle.loads` 只能用于可信数据；外部数据用 `json`。
27. 口令、密钥、连接串不写进代码，统一走配置/环境变量。
28. 文件路径来自外部输入时，用 `Path.resolve()` 后校验是否越出根目录。

## 六、并发

29. 共享可变状态必须加锁（`threading.Lock`）；GIL 不能保证复合操作的原子性。
30. I/O 密集用 `asyncio` 或线程池，CPU 密集用 `ProcessPoolExecutor`。

## 七、常见陷阱

31. 禁止用可变对象（list / dict / set）做参数默认值 —— 默认值只在函数定义时求值
    一次，会跨调用累积。改成 `def f(bucket=None): bucket = bucket if bucket is not None else []`。
32. 不要在函数里原地修改传入的可变参数；确实需要就先 `copy()`，否则调用方的数据被悄悄改掉。
33. `is` 只用于 `None` / `True` / `False` / 单例；数值与字符串比较一律用 `==`。
"""

# references/javascript_rules.md —— 同理，只有审查 JS 时才读。
JAVASCRIPT_RULES_MD = """# JavaScript / TypeScript 代码审查规范（30 条）

> 本文件只在审查 JavaScript / TypeScript 代码时才需要读入上下文。

## 一、变量与作用域

1. 默认用 `const`；需要重新赋值才用 `let`；永远不要用 `var`。
2. 用 `===` / `!==`，不要用 `==` / `!=`（隐式类型转换是 bug 温床）。
3. 变量声明放在使用点附近，不要把所有声明堆在函数开头。
4. 避免在块级作用域外泄漏变量（尤其 `for` 循环里的闭包）。

## 二、函数与异步

5. 优先箭头函数；需要 `this` 动态绑定时才用 `function`。
6. 参数超过 3 个时改用「选项对象」解构。
7. `async` 函数里每个 `await` 都要能抛出可处理的错误。
8. 不要在 `forEach` 里 `await` —— 用 `for...of`，否则不会等待。
9. 并发请求用 `Promise.all` / `allSettled`，不要串行 `await` 拖慢。
10. 永远不要忘记 `catch`；`Promise` 链结尾必须有 `.catch()`。

## 三、错误处理

11. `throw` 只抛 `Error` 及其子类，不要抛字符串。
12. 自定义错误继承 `Error` 并设置 `this.name`，否则堆栈里认不出来。
13. 顶层用 try/catch 包住事件回调，防止一个异常打断整个流程。

## 四、性能

14. 循环内不要做 DOM 查询；先缓存到变量。
15. 频繁触发的 `scroll` / `resize` / `input` 要防抖（debounce）或节流（throttle）。
16. 大数组用 `map` / `filter` / `reduce`，但不要串成五层链式调用 —— 可读性优先。
17. 避免在渲染函数里创建新对象/新函数（会破坏 memo 优化）。

## 五、安全

18. 不要用 `innerHTML` 渲染外部数据；用 `textContent` 或框架的转义机制。
19. 不要 `eval` / `new Function`。
20. `postMessage` 必须校验 `event.origin`。
21. 敏感 token 不要放 `localStorage`，优先 httpOnly Cookie。

## 六、TypeScript 专项

22. 禁止 `any`；实在不确定用 `unknown` 再收窄。
23. 接口用 `interface`，联合/交叉类型用 `type`。
24. 函数返回值显式标注，尤其是导出函数。
25. 用可选链 `?.` 和空值合并 `??`，不要写 `a && a.b && a.b.c`。

## 七、工程化

26. 文件不超过 400 行；超了就按职责拆分。
27. 一个文件只做一件事；`index.js` 只做再导出。
28. 依赖锁文件（`package-lock.json` / `pnpm-lock.yaml`）必须提交。
29. 不要提交 `console.log`；用统一的 logger。
30. 涉及金额、时间、ID 的运算，注意浮点精度与字符串/数字混用。
"""

### 5.2 落盘：写到本课临时目录

这一格把上面三份内容真的写到磁盘上，然后打印目录树。

落盘时两处细节值得留意：

- 写成 **dict** 而不是三次 `write_text`：dict 的键就是技能目录的结构，
  「技能包由哪几个文件组成」一眼可见；
- `newline="\n"`：技能文件是给模型读的纯文本，固定 LF，避免 Windows 下写成 CRLF。

In [ ]:
# ================================================================
# 1. 落盘：把上面三份内容真的写到磁盘上
# ================================================================
def stage_write_files() -> None:
    print("=" * 78)
    print("1. 生成技能文件（真实写盘）")
    print("=" * 78)

    BUILD_REFERENCES_DIR.mkdir(parents=True, exist_ok=True)
    # 本技能要落盘的三个文件。写成 dict 而不是三次 write_text，
    # 是为了让「技能包由哪几个文件组成」一眼可见 —— dict 的键就是技能目录的结构。
    # 注意 references/ 里的两份规范【必须】分文件放：合成一份的话，
    # 「只读命中的那一份」就无从谈起了，渐进式披露也就没了。
    # notebook 版：这里写的是 BUILD_*（WORKDIR 里的本课专属子目录），不是仓库里那份已跟踪的技能。
    files = {
        BUILD_SKILL_DIR / "SKILL.md": SKILL_MD,
        BUILD_REFERENCES_DIR / "python_rules.md": PYTHON_RULES_MD,
        BUILD_REFERENCES_DIR / "javascript_rules.md": JAVASCRIPT_RULES_MD,
    }
    for path, content in files.items():
        # newline="\n"：技能文件是给模型读的纯文本，固定 LF，避免 Windows 下 CRLF
        path.write_text(content, encoding="utf-8", newline="\n")
        print(f"  已写入 {path.relative_to(WORKDIR)}  ({len(content)} 字符)")

    print()
    print("  生成后的目录树：")
    _print_tree(BUILD_ROOT)


def _print_tree(root: Path) -> None:
    """打印目录树（只用于观察，逻辑刻意写得很朴素）。"""

    def walk(path: Path, prefix: str = "") -> None:
        children = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name))
        for index, child in enumerate(children):
            is_last = index == len(children) - 1
            branch = "└── " if is_last else "├── "
            if child.is_dir():
                print(f"    {prefix}{branch}{child.name}/")
                walk(child, prefix + ("    " if is_last else "│   "))
            else:
                text = child.read_text(encoding="utf-8")
                print(f"    {prefix}{branch}{child.name}    # {len(text.splitlines())} 行")

    print(f"    {root.name}/")
    walk(root)


stage_write_files()

### 预期输出

```text
==============================================================================
1. 生成技能文件（真实写盘）
==============================================================================
  已写入 skill_build\code-review-skill\SKILL.md  (783 字符)
  已写入 skill_build\code-review-skill\references\python_rules.md  (1771 字符)
  已写入 skill_build\code-review-skill\references\javascript_rules.md  (1411 字符)

  生成后的目录树：
    skill_build/
    └── code-review-skill/
        ├── references/
        │   ├── javascript_rules.md    # 54 行
        │   └── python_rules.md    # 58 行
        └── SKILL.md    # 37 行
```

三个字符数（783 / 1771 / 1411）请记一下 —— 下一格比对的就是它们，
第 5.7 节的 Token 账本也用的这几个数。

### 5.3 与仓库里那份**已跟踪**的技能包逐字节比对

这一格是 notebook 版新增的自证步骤：把刚写进临时目录的三个文件，与
`Agent/08_skills/skills/code-review-skill/` 下**仓库已跟踪**的同名文件逐字节比对。

结果必须是「三份全一致」—— 这说明：

1. 本 notebook 引用的那份现成技能，和源文件里的常量**同源**（当年就是它写出来的）；
2. 本 notebook 不需要、也没有往章节目录里写任何文件，`git status` 应当保持干净。

In [ ]:
# ===== 与仓库里已跟踪的那一份逐字节比对（本 notebook 不重写它）=====
print("=" * 78)
print("2. 与仓库里已跟踪的技能包比对（确认同源，且没有改动它）")
print("=" * 78)

_pairs = [
    (BUILD_SKILL_DIR / "SKILL.md", SKILL_DIR / "SKILL.md"),
    (BUILD_REFERENCES_DIR / "python_rules.md", REFERENCES_DIR / "python_rules.md"),
    (BUILD_REFERENCES_DIR / "javascript_rules.md", REFERENCES_DIR / "javascript_rules.md"),
]
_all_same = True
for _built, _tracked in _pairs:
    if not _tracked.is_file():
        print(f"  ✗ 仓库里缺少 {_tracked.name}（跳过比对）")
        _all_same = False
        continue
    _a = _built.read_text(encoding="utf-8")
    _b = _tracked.read_text(encoding="utf-8")
    _same = _a == _b
    _all_same = _all_same and _same
    print(f"  {'✅ 一致  ' if _same else '✗ 不一致'} {_built.name:<20}"
          f"临时 {len(_a)} 字符 / 仓库 {len(_b)} 字符")

print()
print("  结论：", "三份全部一致 —— notebook 引用的就是仓库里那份现成技能，没有另造一份。"
      if _all_same else "存在差异，请检查本节源码常量是否被改动过。")
print()
print("  仓库里真实存在的技能目录（跟踪文件，只读）：")
_print_tree(SKILLS_ROOT)

### 预期输出

```text
==============================================================================
2. 与仓库里已跟踪的技能包比对（确认同源，且没有改动它）
==============================================================================
  ✅ 一致   SKILL.md            临时 783 字符 / 仓库 783 字符
  ✅ 一致   python_rules.md     临时 1771 字符 / 仓库 1771 字符
  ✅ 一致   javascript_rules.md 临时 1411 字符 / 仓库 1411 字符

  结论： 三份全部一致 —— notebook 引用的就是仓库里那份现成技能，没有另造一份。

  仓库里真实存在的技能目录（跟踪文件，只读）：
    skills/
    └── code-review-skill/
        ├── references/
        │   ├── javascript_rules.md    # 54 行
        │   └── python_rules.md    # 58 行
        └── SKILL.md    # 37 行
```

三行全是「✅ 一致」就说明：本 notebook 跑完 **`git status` 应当保持干净**
（唯一写盘的地方在 `tmp_nb_work/skill_build/`，已被 `.gitignore` 覆盖）。

### 5.4 最小 YAML front-matter 解析器（只依赖标准库 `re`）

这是本课唯一一处「值得盯着看」的正则，逐段拆开：

| 片段 | 作用 |
|---|---|
| `\A` | 从文件**开头**匹配（不是从第一行）—— 前面有空行就解析失败，这就是「`---` 必须是第 1 行」的机器校验版 |
| `---[ \t]*\r?\n` | 第一行是三个减号；`[ \t]*` 允许行尾有空格，`\r?\n` 同时兼容 LF 与 CRLF |
| `(.*?)` | 捕获组 1 = front-matter 正文；**非贪婪**，所以只吃到第一个单独的 `---` 为止 |
| `\r?\n---[ \t]*\r?\n?` | 闭合的 `---` 行；末尾 `\r?\n?` 用 `?` 是因为文件可能恰好以 `---` 结尾 |
| `(.*)\Z` | 捕获组 2 = Markdown 正文，一直吃到文件结尾 |

**它不支持什么**（`parse_front_matter` 的 docstring 里也写了）：多行字符串（`description: |`）、
列表（`- x`）、锚点、流式语法。本项目自己生成的 `SKILL.md` 只用「空格分隔的标量」
和「一层缩进 map」两种形状，够用即止 —— **不假装是 YAML 解析器**。

In [ ]:
# ================================================================
# 2. 最小 YAML front-matter 解析器（只依赖标准库 re）
# ================================================================
import re

# 匹配文件开头的 `--- ... ---`，并把后面的正文一起捕获。
# 注意用 re.DOTALL，否则 `.` 不匹配换行，多行 front-matter 就抓不到。
_FRONT_MATTER_RE = re.compile(r"\A---[ \t]*\r?\n(.*?)\r?\n---[ \t]*\r?\n?(.*)\Z", re.DOTALL)


def parse_front_matter(text: str) -> tuple[dict, str]:
    """把 SKILL.md 拆成 (元数据字典, 正文)。

    支持的 YAML 子集：
        key: value                     → 顶层标量
        key:                           → 顶层 map，缩进两格的子键
          sub: value
    不支持：多行字符串、列表 `- x`、锚点、流式语法。
    本项目自己生成的 SKILL.md 用的就是上面两种形状，够用。

    ⚠️ 这不是通用 YAML 解析器，是「够用即止」的窄解析器：
       front-matter 里写了列表（`allowed-tools:\n  - read_file`）或
       块标量（`description: |`），这里会静默丢掉那一部分 —— 所以本项目的
       SKILL.md 一律用「空格分隔的标量」或「一层缩进 map」这两种形状。
    """
    match = _FRONT_MATTER_RE.match(text)
    if not match:
        # 报错信息直接告诉读者「文件应该长什么样」，比抛一个 NoneType 崩溃有用得多
        raise ValueError("SKILL.md 必须以第 1 行的 `---` 开始，且以单独一行的 `---` 结束")

    raw_meta, body = match.group(1), match.group(2)   # 组 1 = 元数据，组 2 = 正文
    meta: dict = {}
    # current_key 记录「上一个没有值的顶层 key」。YAML 靠缩进表达父子关系，
    # 而逐行解析时每读到一行缩进内容，必须知道「我是谁的孩子」—— 就是它。
    current_key: str | None = None

    for line in raw_meta.splitlines():
        # 空行与 `#` 注释直接跳过
        if not line.strip() or line.lstrip().startswith("#"):
            continue

        if line[0] in " \t":
            # 缩进行 → 属于上一个 key 的嵌套 map
            if current_key is None:
                # 开头就来缩进行（没有父 key）→ 非法 YAML，忽略而不是崩
                continue
            # 父 key 此刻必须是个 dict；若它之前被解析成标量，就地升级成 dict
            # （YAML 里同名 key 既当标量又当 map 是错的，但这里选择宽容处理）
            if not isinstance(meta.get(current_key), dict):
                meta[current_key] = {}
            # partition(":") 而不是 split(":")：只按【第一个】冒号切一刀，
            # 这样值里带冒号（比如 URL、时间 12:30）也不会被切坏
            sub_key, _, sub_value = line.strip().partition(":")
            # 顺手剥掉 YAML 里常见的引号：version: "1.0.0" → 1.0.0
            meta[current_key][sub_key.strip()] = sub_value.strip().strip('"').strip("'")
            continue

        key, sep, value = line.partition(":")
        if not sep:
            # 整行没有冒号 → 不是合法的 key: value，忽略（宽容策略，不抛异常）
            continue
        key = key.strip()
        value = value.strip()
        if value:
            # 顶层标量：顺手去掉 YAML 里常见的引号
            meta[key] = value.strip('"').strip("'")
            # 标量没有子键，后面若来缩进行就不该再挂到它下面
            current_key = None
        else:
            # `key:` 后面没有值 → 准备接收缩进的子键
            meta[key] = {}
            current_key = key

    return meta, body


def as_list(value: str | dict | None) -> list[str]:
    """把 `trigger_keywords: a b c` 这类空格/逗号分隔的标量转成列表。"""
    if not isinstance(value, str):
        return []
    return [item for item in re.split(r"[\s,，、]+", value) if item]

先单独验一下解析器（不在源文件里，属于 notebook 版的便利步骤）：
拿真实的 `SKILL.md` 走一遍，看看 front-matter 被拆成了什么。

In [ ]:
# ===== 解析器单跑一遍（notebook 版便利步骤）=====
_demo_text = (SKILL_DIR / "SKILL.md").read_text(encoding="utf-8") if SKILLS_READY else SKILL_MD
_meta, _body = parse_front_matter(_demo_text)
print("解析出的 name        :", _meta.get("name"))
print("解析出的 license     :", _meta.get("license"))
print("解析出的 metadata    :", _meta.get("metadata"))
print("解析出的 allowed-tools:", as_list(_meta.get("allowed-tools")))
print("正文行数             :", len(_body.splitlines()))
print("正文第一行           :", _body.splitlines()[0] if _body.splitlines() else "（空）")

### 预期输出

```text
解析出的 name        : code-review-skill
解析出的 license     : MIT
解析出的 metadata    : {'author': 'jxsd', 'version': '1.0.0'}
解析出的 allowed-tools: ['read_file', 'write_file']
正文行数             : 26
正文第一行           : 
```

三处对照着看：

1. `metadata` 被解析成了**嵌套 dict**（`author` / `version` 两个子键）——
   这就是「一层缩进 map」那条分支在工作，`version: "1.0.0"` 的引号也被剥掉了；
2. `allowed-tools` 原始值是空格分隔的字符串，`as_list()` 把它切成 `['read_file', 'write_file']`；
3. **正文第一行是空的** —— front-matter 闭合那个 `---` 后面紧跟一个换行，
   所以正文以 `\n` 开头。这不是 bug，真要严谨可以 `body.lstrip("\n")`；
   但 `SKILL.md` 正文前留一个空行本来就更好读。

### 5.5 第一步「选择」：只读 front-matter

`stage_select()` 遍历 `SKILLS_ROOT/*/SKILL.md`，对每个技能**只解析元数据**，
然后算出两个数：常驻上下文的 `name+description` 字符数，以及 `SKILL.md` 全文的字符数。
两者的差值，就是「只读元数据」替 agent 省下的部分。

> ⚠️ 注意：真实 deepagents 在这条路径上会去 **`ls` 技能父目录 + 下载每个 `SKILL.md`**；
> 这里用本地 `glob` 是为了让演示结果**完全可复现**（不依赖模型、不依赖网络）。

In [ ]:
# ================================================================
# 3. 第一步「选择」：只读 front-matter，正文一个字都不进上下文
# ================================================================
def stage_select() -> list[dict]:
    print()
    print("=" * 78)
    print("2. 第一步「选择」：AI 只读取每个技能的名称和描述（只有几十字符）")
    print("=" * 78)

    skills: list[dict] = []
    for skill_md in sorted(SKILLS_ROOT.glob("*/SKILL.md")):
        full_text = skill_md.read_text(encoding="utf-8")
        meta, body = parse_front_matter(full_text)

        # 这是关键对比：常驻上下文的是 name+description，
        # 而 read_file 命中的才是全文（正文 + 后续 reference）。
        meta_chars = len(meta.get("name", "")) + len(meta.get("description", ""))
        full_chars = len(full_text)
        full_lines = len(full_text.splitlines())

        print(f"  发现技能：{meta.get('name')}")
        print(f"    description : {meta.get('description')}")
        print(f"    license     : {meta.get('license')}")
        print(f"    metadata    : {meta.get('metadata')}")
        print(f"    allowed-tools: {as_list(meta.get('allowed-tools'))}")
        print(
            f"    常驻上下文字符数（name+description）= {meta_chars} 字符；"
            f"SKILL.md 全文 = {full_lines} 行 / {full_chars} 字符"
        )
        print(f"    → 只读元数据可省下 {full_chars - meta_chars} 字符（{100 * (1 - meta_chars / full_chars):.1f}%）")
        print()

        skills.append({"path": skill_md, "meta": meta, "body": body, "full_text": full_text})

    if not skills:
        print("  （没有发现任何技能，请先确认上面的写盘步骤是否成功）")
    return skills


# 只解析元数据，不读正文 —— 这一步对应 agent 启动时的「技能选择」
all_skills = stage_select()

### 预期输出

```text
==============================================================================
2. 第一步「选择」：AI 只读取每个技能的名称和描述（只有几十字符）
==============================================================================
  发现技能：code-review-skill
    description : 审查 Python / JavaScript 代码，检查规范性、性能与安全问题。当用户要求「代码审查 / review / 检查代码 / 优化这段代码」时使用。
    license     : MIT
    metadata    : {'author': 'jxsd', 'version': '1.0.0'}
    allowed-tools: ['read_file', 'write_file']
    常驻上下文字符数（name+description）= 97 字符；SKILL.md 全文 = 37 行 / 783 字符
    → 只读元数据可省下 686 字符（87.6%）

```

**97 字符** vs **783 字符** —— 这就是「选择」这一步的全部意义：
agent 启动时，这个技能只贡献 97 个字符到系统提示词里。技能越多，
省下的绝对量越大（100 个技能 ≈ 常驻 9700 字符，而不是 78000 字符）。

### 5.6 第二步「学习」：命中才读 `SKILL.md` 全文

用户输入：`"帮我审查这段 Python 代码，看看规范性和性能问题"`。

`stage_learn()` 用**极简的关键词打分**判断命中（`description` + `trigger_keywords`
里出现一个词就算命中），命中后打印 `SKILL.md` 正文。

> ⚠️ 真实 deepagents **不跑这套算法** —— 它把 `name`/`description` 交给大模型，
> 由模型自己决定用不用。这里用确定性规则，是为了让演示结果**可复现**。

In [ ]:
# ================================================================
# 4. 第二步「学习」：命中技能后，才把 SKILL.md 全文加载进来
# ================================================================
def stage_learn(skills: list[dict], user_query: str) -> dict | None:
    print("=" * 78)
    print("3. 第二步「学习」：匹配到技能，才把 SKILL.md 正文加载进来")
    print("=" * 78)
    print(f"  用户输入：{user_query}")

    hit = None
    for skill in skills:
        meta = skill["meta"]
        # 极简的关键词打分：description + trigger_keywords 里出现一个就算命中。
        # 真实的 deepagents 不跑这套算法 —— 它把 name/description 交给大模型，
        # 由模型自己决定用不用。这里用确定性规则是为了让演示结果可复现。
        keywords = as_list(meta.get("trigger_keywords")) + [
            token for token in re.split(r"[，。、；/\s]+", str(meta.get("description", ""))) if len(token) >= 2
        ]
        matched = [kw for kw in keywords if kw and kw in user_query]
        print(f"    候选技能 {meta.get('name')}：命中关键词 {matched or '（无）'}")
        if matched and hit is None:
            hit = skill

    if hit is None:
        print("  没有技能命中 → 一个字的正文都不会加载（这就是省 Token 的来源）")
        return None

    meta = hit["meta"]
    print()
    print(f"  ✅ 命中技能「{meta.get('name')}」，加载 {hit['path'].name} 全文：")
    print(f"     正文 {len(hit['body'].splitlines())} 行 / {len(hit['body'])} 字符"
          f"（对比：常驻的 name+description 只有 "
          f"{len(str(meta.get('name'))) + len(str(meta.get('description')))} 字符）")
    print("  " + "-" * 74)
    for line in hit["body"].splitlines():
        print("  | " + line)
    print("  " + "-" * 74)
    return hit


query = "帮我审查这段 Python 代码，看看规范性和性能问题"
hit_skill = stage_learn(all_skills, query)

### 预期输出

```text
==============================================================================
3. 第二步「学习」：匹配到技能，才把 SKILL.md 正文加载进来
==============================================================================
  用户输入：帮我审查这段 Python 代码，看看规范性和性能问题
    候选技能 code-review-skill：命中关键词 ['规范', '性能', '审查', 'Python', '代码']

  ✅ 命中技能「code-review-skill」，加载 SKILL.md 全文：
     正文 26 行 / 480 字符（对比：常驻的 name+description 只有 97 字符）
  --------------------------------------------------------------------------
  | 
  | # 代码审查技能
  | 
  | 你是资深代码审查专家。收到审查请求后，严格按下面的流程执行。
  | 
  | ## 第 1 步：判断语言
  | 
  | - 代码是 Python（`.py`，或含 `def` / `import` / 缩进块）→ 读 `references/python_rules.md`
  | - 代码是 JavaScript / TypeScript（`.js` `.ts`，或含 `const` / `=>` / `function`）→ 读 `references/javascript_rules.md`
  | - 混合项目 → 两种规范都读，分别给出结论
  | 
  | **只读命中的那一份规范**，不要为了「保险」把两份都读进来。
  | 
  | ## 第 2 步：逐条比对
  | 
  | 按规范的编号逐条检查，每条给出三样东西：
  | 
  | 1. 结论：通过 / 不通过 / 存疑
  | 2. 证据：`文件名:行号` + 原始代码片段
  | 3. 修法：可直接替换的改法，不要写「建议优化」这种空话
  | 
  | ## 第 3 步：汇总
  | 
  | 输出一张表（严重 / 一般 / 建议三档），最后给一句总体结论。
  | 
  | 不要重写整个文件，只给需要改的片段。
  --------------------------------------------------------------------------
```

命中关键词那一行值得盯一下：`['规范', '性能', '审查', 'Python', '代码']` 里
**没有 `代码审查`、也没有 `检查代码`** —— 因为用户那句里根本没出现这两个词。
`trigger_keywords` 不是「命中其一就必中」，而是「在用户输入里逐个找」；
真正让这次命中的是 `description` 里的普通词（`规范`/`性能`/`审查`/`代码`）。

另外注意 `Python` 是**大小写敏感**匹配（`kw in user_query`），
所以用户输入里的 `Python` 必须大写才能命中这个关键词。

### 5.7 第三步「使用」：只读命中的那一个 `references/` 文件

`stage_use()` 先从 `SKILL.md` 正文里把 `references/xxx.md` 抓出来（相当于模型
`read_file` 时的候选集），再按目标语言**只加载命中的那一份**，最后打印 Token 账本：

```text
    references/ 下全部参考文件合计 : NNNN 字符
    本次实际加载                  : SKILL.md 正文 NNN 字符 + NNN 字符 = NNN 字符
    节省                          : NNNN 字符（XX.X% 的参考内容没有进上下文）
```

这一行「节省」就是渐进式披露在**数字上**的样子。

In [ ]:
# ================================================================
# 5. 第三步「使用」：按 SKILL.md 的指示，只读命中的那个 reference
# ================================================================
def stage_use(skill: dict | None, user_query: str) -> None:
    print()
    print("=" * 78)
    print("4. 第三步「使用」：按 SKILL.md 的指示，只读命中的那个 reference 文件")
    print("=" * 78)

    if skill is None:
        print("  上一步没有命中技能，这里无事可做。")
        return

    # SKILL.md 正文里用相对路径点名了 references/xxx.md，
    # 这里把它们抓出来 —— 相当于模型 read_file 时的候选集。
    declared = re.findall(r"references/[\w./-]+\.md", skill["body"])
    print(f"  SKILL.md 正文里点名的参考文件：{sorted(set(declared))}")

    # 目标语言的判断：和 SKILL.md 里写的规则一致（Python → python_rules.md，以此类推）
    if "python" in user_query.lower() or "py" in user_query.lower():
        target_name = "python_rules.md"
    elif "javascript" in user_query.lower() or "js" in user_query.lower() or "ts" in user_query.lower():
        target_name = "javascript_rules.md"
    else:
        target_name = None

    all_refs = sorted(REFERENCES_DIR.glob("*.md"))
    total_all = sum(len(p.read_text(encoding="utf-8")) for p in all_refs)
    loaded_chars = 0

    for ref in all_refs:
        if target_name is not None and ref.name != target_name:
            print(f"  ✗ 跳过 {ref.name}（本次是 {target_name} 的任务，不加载）"
                  f"  —— 省下 {len(ref.read_text(encoding='utf-8'))} 字符")
            continue
        text = ref.read_text(encoding="utf-8")
        loaded_chars += len(text)
        preview = text.splitlines()[:6]
        print(f"  ✅ 加载 {ref.name}（{len(text.splitlines())} 行 / {len(text)} 字符），前几行：")
        for line in preview:
            print("      " + line)
        print("      ...")

    print()
    print("  【Token 账本】")
    print(f"    references/ 下全部参考文件合计 : {total_all} 字符")
    print(f"    本次实际加载                  : SKILL.md 正文 {len(skill['body'])} 字符"
          f" + {loaded_chars} 字符 = {len(skill['body']) + loaded_chars} 字符")
    print(f"    节省                          : {total_all - loaded_chars} 字符"
          f"（{100 * (1 - loaded_chars / total_all):.1f}% 的参考内容没有进上下文）")


stage_use(hit_skill, query)

### 预期输出

```text
==============================================================================
4. 第三步「使用」：按 SKILL.md 的指示，只读命中的那个 reference 文件
==============================================================================
  SKILL.md 正文里点名的参考文件：['references/javascript_rules.md', 'references/python_rules.md']
  ✗ 跳过 javascript_rules.md（本次是 python_rules.md 的任务，不加载）  —— 省下 1411 字符
  ✅ 加载 python_rules.md（58 行 / 1771 字符），前几行：
      # Python 代码审查规范（33 条）

      > 本文件只在审查 Python 代码时才需要读入上下文 —— 这就是「渐进式披露」。

      ## 一、命名与风格（PEP 8）

      ...

  【Token 账本】
    references/ 下全部参考文件合计 : 3182 字符
    本次实际加载                  : SKILL.md 正文 480 字符 + 1771 字符 = 2251 字符
    节省                          : 1411 字符（44.3% 的参考内容没有进上下文）
```

**44.3%** 就是渐进式披露在这个小技能上的收益。它是按「参考文件」算的：
本次真正进上下文的参考内容只有 `python_rules.md` 那 1771 字符，
`javascript_rules.md` 的 1411 字符永远没被读进来。

### 5.8 换一个查询：对称效果

源文件注释里特意提了这件事：把查询换成 JavaScript，就能看到**完全对称**的结果
—— 这次加载的是 `javascript_rules.md`，而 `python_rules.md` 被跳过。

**真实交互时请直接改这一格里的 `js_query`**，重跑即可。

In [ ]:
# ===== 对称验证：换成 JS 查询 =====
js_query = "帮我 review 这段 javascript 代码"
js_hit = stage_learn(all_skills, js_query)
stage_use(js_hit, js_query)

### 预期输出

```text
==============================================================================
3. 第二步「学习」：匹配到技能，才把 SKILL.md 正文加载进来
==============================================================================
  用户输入：帮我 review 这段 javascript 代码
    候选技能 code-review-skill：命中关键词 ['review', '代码', 'review']

  ✅ 命中技能「code-review-skill」，加载 SKILL.md 全文：
     正文 26 行 / 480 字符（对比：常驻的 name+description 只有 97 字符）

……（`SKILL.md` 正文那 26 行与 5.6 节逐字相同，此处略）

==============================================================================
4. 第三步「使用」：按 SKILL.md 的指示，只读命中的那个 reference 文件
==============================================================================
  SKILL.md 正文里点名的参考文件：['references/javascript_rules.md', 'references/python_rules.md']
  ✅ 加载 javascript_rules.md（54 行 / 1411 字符），前几行：
      # JavaScript / TypeScript 代码审查规范（30 条）

      > 本文件只在审查 JavaScript / TypeScript 代码时才需要读入上下文。

      ## 一、变量与作用域

      ...
  ✗ 跳过 python_rules.md（本次是 javascript_rules.md 的任务，不加载）  —— 省下 1771 字符

  【Token 账本】
    references/ 下全部参考文件合计 : 3182 字符
    本次实际加载                  : SKILL.md 正文 480 字符 + 1411 字符 = 1891 字符
    节省                          : 1771 字符（55.7% 的参考内容没有进上下文）
```

与 5.6 / 5.7 完全对称：**同一个技能、同一段代码，换个查询，进上下文的就是另一份规范**。
命中关键词里 `review` 出现了两次 —— 一次来自 `trigger_keywords`，一次来自 `description`
的分词，打分算法没去重，所以列表里会重复。

> ⚠️ 上面是**节选**：`SKILL.md` 正文那 26 行与 5.6 节逐字相同，用「……」略去了。
> **别逐字比对整段** —— 该核的是「命中关键词 → 加载哪份参考文件 → 省下多少字符」这条链路。

## 6. 小结（概念篇）

这一格是源文件 `01_概念与原理_jxsd.py` 的 `section_summary()`：
把「一个 Skill 是什么」压成七行，再报一下技能目录当前的状态。

In [ ]:
# ================================================================
# 5. 小结
# ================================================================
def section_summary() -> None:
    print("=" * 78)
    print("5. 小结")
    print("=" * 78)
    print(
        f"""
    一个 Skill = 一个目录 + 一个 SKILL.md。
    目录名 = name；YAML front-matter = 元数据；正文 = 指令。
    references/ 放「需要时才读」的文档，scripts/ 放「让 agent 去跑」的脚本，
    assets/ 放「给脚本用的」资源。

    加载三步：选择（只读 name+description）→ 学习（命中后读 SKILL.md 全文）
              → 使用（按指示读参考文件 / 跑脚本）。

    本节的目录树与字段表都是打印出来的常量。真正的技能文件由下一节生成：
        {SKILLS_ROOT}
    当前状态：{"已存在" if SKILLS_ROOT.exists() else "尚未生成（请运行下一本 02_三种技能载体.ipynb）"}
"""
    )


section_summary()

### 预期输出

```text
==============================================================================
5. 小结
==============================================================================

    一个 Skill = 一个目录 + 一个 SKILL.md。
    目录名 = name；YAML front-matter = 元数据；正文 = 指令。
    references/ 放「需要时才读」的文档，scripts/ 放「让 agent 去跑」的脚本，
    assets/ 放「给脚本用的」资源。

    加载三步：选择（只读 name+description）→ 学习（命中后读 SKILL.md 全文）
              → 使用（按指示读参考文件 / 跑脚本）。

    本节的目录树与字段表都是打印出来的常量。真正的技能文件由下一节生成：
        F:\ProGram\Python_Base\Agent\08_skills\skills
    当前状态：已存在

```

最后那行「当前状态：已存在」就是第 5.3 节逐字节比对过的那个目录 ——
本 notebook 从第 0.2 节起就只读它，一个字都没往那儿写。

## 7. 小结（实战篇）：本节演示的三步 ↔ 真实框架里的代码

这一格是源文件 `02_SKILL示例_jxsd.py` 的 `section_summary()`（本节里重命名为
`section_summary_hands_on` 以区分概念篇那份）。它把「本 notebook 做的事」和
「真实 deepagents 里对应的实现」并排放：

| 本节做的事（纯标准库、结果可复现） | 真实 deepagents 里对应的实现 |
|---|---|
| 选择：`glob('*/SKILL.md')` + 只解析 front-matter | `SkillsMiddleware.before_agent` 遍历 skills 源目录 → `download_files` 取每个 `SKILL.md` → 只留元数据拼进系统提示词 |
| 学习：关键词命中后打印 `SKILL.md` 正文 | 模型自己调 `read_file(SKILL.md, limit=1000)` |
| 使用：按正文里的 `references/` 路径读 | 模型自己决定再 `read_file` 哪个参考文件 |

**结论**：技能作者要做的，就是把「什么时候读哪个文件」写清楚 ——
框架只负责把 `name`/`description` 摆在模型面前，剩下的靠模型的判断力，
以及你在 `SKILL.md` 里写的路由规则。

In [ ]:
# ================================================================
# 6. 小结 / 与 deepagents 的对应关系
# ================================================================
def section_summary_hands_on() -> None:
    print()
    print("=" * 78)
    print("5. 小结：本节演示的三步，对应真实框架里的哪段代码")
    print("=" * 78)
    print(
        f"""
    本节做的事情（纯标准库、结果可复现）        真实 deepagents 里对应的实现
    ------------------------------------------------------------------------
    1. 选择：glob('*/SKILL.md') + 只解析 front-matter
                                             SkillsMiddleware.before_agent
                                             遍历 skills 源目录 → download_files
                                             取每个 SKILL.md → 只留元数据拼进系统提示词
    2. 学习：关键词命中后打印 SKILL.md 正文 模型自己调 read_file(SKILL.md, limit=1000)
    3. 使用：按正文里的 references/ 路径读模型自己决定再 read_file 哪个参考文件

    结论：技能作者要做的，就是把「什么时候读哪个文件」写清楚 ——
    框架只负责把 name/description 摆在模型面前，剩下的靠模型的判断力，
    以及你在 SKILL.md 里写的路由规则。

    本节引用/比对的技能目录（仓库已跟踪，只读）：
        {SKILLS_ROOT}
    本 notebook 的落盘副本在临时目录（不进版本库）：
        {BUILD_ROOT}
    下一本 02_三种技能载体.ipynb 会把它装进 deepagents / PostgreSQL / Claude Code。
"""
    )


section_summary_hands_on()

### 预期输出

```text
==============================================================================
5. 小结：本节演示的三步，对应真实框架里的哪段代码
==============================================================================

    本节做的事情（纯标准库、结果可复现）        真实 deepagents 里对应的实现
    ------------------------------------------------------------------------
    1. 选择：glob('*/SKILL.md') + 只解析 front-matter
                                             SkillsMiddleware.before_agent
                                             遍历 skills 源目录 → download_files
                                             取每个 SKILL.md → 只留元数据拼进系统提示词
    2. 学习：关键词命中后打印 SKILL.md 正文 模型自己调 read_file(SKILL.md, limit=1000)
    3. 使用：按正文里的 references/ 路径读模型自己决定再 read_file 哪个参考文件

    结论：技能作者要做的，就是把「什么时候读哪个文件」写清楚 ——
    框架只负责把 name/description 摆在模型面前，剩下的靠模型的判断力，
    以及你在 SKILL.md 里写的路由规则。

    本节引用/比对的技能目录（仓库已跟踪，只读）：
        F:\ProGram\Python_Base\Agent\08_skills\skills
    本 notebook 的落盘副本在临时目录（不进版本库）：
        F:\ProGram\Python_Base\Agent\08_skills\tmp_nb_work\skill_build
    下一本 02_三种技能载体.ipynb 会把它装进 deepagents / PostgreSQL / Claude Code。

```

两行路径正好对应本 notebook 的两条数据流：**读**的是仓库里那份已跟踪技能，
**写**的是临时目录里的副本（下一次执行会覆盖它，所以它永远不是「源」）。

## 小结

- **一个 Skill = 一个目录 + 一个 `SKILL.md`**：目录名就是 `name`，
  YAML front-matter 是元数据，正文是给模型的指令；
- **`references/` 是渐进式披露的主战场**：`SKILL.md` 当路由器（十几行），
  规范细节按需要时才读 —— 本节的 Token 账本把这件事变成了具体数字；
- **加载三步**：选择（只读 `name`+`description`）→ 学习（命中才读 `SKILL.md` 全文）
  → 使用（按正文指示读参考文件 / 跑脚本）；
- **三个可选目录分工**：`scripts/` 让 agent 去跑，`references/` 需要时才读，
  `assets/` 给脚本和模型用；
- **`description` 是最关键的字段**：模型只靠它决定要不要用这个技能，
  所以要写「做什么 + 什么时候用 + 关键触发词」。

下一本 `02_三种技能载体.ipynb` 会把这份技能真正**装进**三个载体：
deepagents 的 `create_deep_agent(skills=[...])`、PostgreSQL 云技能、Claude Code 的技能目录。

## 常见坑

1. **`---` 必须是文件第 1 行**，前面有空行就解析失败；正文里也不能出现单独一行的 `---`，
   否则 front-matter 会提前闭合，正文被吃掉（第 5.4 节的正则里 `\A` 就是这条规矩的机器版本）。
2. **`name` 必须等于所在目录名**。deepagents 会校验并在日志里警告
   `name 'x' must match directory name 'y'`。
3. **`trigger_keywords` 在 deepagents 0.7.13 里不生效** —— 中间件只解析
   `name`/`description`/`license`/`compatibility`/`metadata`/`allowed-tools`。
   想提高命中率，把关键词写进 `description`。
4. **两份规范必须分文件放**。合成一份 `all_rules.md`，渐进式披露就退化成全量加载，
   「只读命中的那一份」也就无从谈起。
5. **技能文件固定 LF**（`newline="\n"`）。Windows 下默认写成 CRLF，
   提交后 diff 会全是噪声，且 `\r?\n` 兼容写法只在解析侧兜底。
6. **不要用 `glob` 的顺序做假设**：`Path.iterdir()` / `glob()` 的顺序依赖文件系统，
   要稳定输出就得显式 `sorted(...)`（本节的两个打印函数都排了序）。
7. **notebook 里没有 `__file__`**：源文件的 `Path(__file__).resolve().parent`
   在这里一律换成 `NB_DIR`（本节用 `HERE = NB_DIR` 一行顶掉）。

## 官方链接

- LangChain · Skills（多智能体一章）：<https://docs.langchain.com/oss/python/langchain/multi-agent/skills>
- Deep Agents · Skills：<https://docs.langchain.com/oss/python/deepagents/skills>

> 本节引用的技能资源（仓库已跟踪，只读）：
> `Agent/08_skills/skills/code-review-skill/SKILL.md`
> 与 `Agent/08_skills/skills/code-review-skill/references/{python_rules.md, javascript_rules.md}`。